## Análise de pesquisa_satisfacao - Bronze Layer

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"

df_pesquisa = spark.table(f"{bronze_schema}.pesquisa_satisfacao")
total = df_pesquisa.count()
display(df_pesquisa)

### Funções auxiliares

In [0]:
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"

def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))

def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)

def mostrar(df_, descricao):
    """Imprime a descrição e faz o display. Se o DataFrame estiver vazio, informa que não há casos."""
    if df_.isEmpty():
        print(f"\n{descricao}: nenhum caso encontrado")
    else:
        print(f"\n{descricao}")
        display(df_)

### 1. Duplicidade
Para `id_chamado`, o conteúdo comparado é só a nota. O `id_pesquisa` fica de fora porque muda entre as cópias.

In [0]:
colunas_negocio = [c for c in df_pesquisa.columns if c != "ingestion_datetime"]
distintas = df_pesquisa.select(*colunas_negocio).distinct().count()
print(f"Linhas excedentes por cópia exata: {total - distintas}")

# chave -> colunas comparadas para saber se as linhas repetidas têm conteúdo diferente
chaves = {
    "id_pesquisa": ["id_chamado", "nota_atendimento"],
    "id_chamado": ["nota_atendimento"],
}
for chave, conteudo in chaves.items():
    repetidos = (
        df_pesquisa.filter(~eh_faltante(chave))
          .groupBy(texto(chave).alias(chave))
          .agg(
              F.count("*").alias("ocorrencias"),
              F.size(F.collect_set(F.struct(*conteudo))).alias("versoes_distintas"),
          )
          .filter(F.col("ocorrencias") > 1)
    )
    n_ids = repetidos.count()
    linhas_envolvidas = repetidos.agg(F.sum("ocorrencias")).first()[0] or 0
    n_divergentes = repetidos.filter(F.col("versoes_distintas") > 1).count()
    print(f"\n{chave} repetidos: {n_ids} ids em {linhas_envolvidas} linhas ({linhas_envolvidas - n_ids} linhas excedentes)")
    print(f"  -> {n_divergentes} desses ids aparecem com conteúdo diferente entre as linhas")
    mostrar(repetidos.orderBy(F.desc("versoes_distintas"), F.desc("ocorrencias"), chave),
            f"{chave} repetidos (ocorrencias = nº de linhas; versoes_distintas = nº de conteúdos diferentes)")
    if chave == "id_chamado":
        ids_chamado_repetidos = repetidos

In [0]:
# linhas completas dos id_chamado com mais de uma pesquisa (join sobre o id com o mesmo trim usado no agrupamento)
linhas_repetidas = (
    df_pesquisa
      .withColumn("id_chave", texto("id_chamado"))
      .join(ids_chamado_repetidos.select(F.col("id_chamado").alias("id_chave")), on="id_chave", how="inner")
      .drop("id_chave")
      .orderBy("id_chamado", "id_pesquisa")
)
mostrar(linhas_repetidas, "Linhas completas dos id_chamado com mais de uma pesquisa")

### 2. Entradas faltantes por coluna

In [0]:
exprs = []
for c in df_pesquisa.columns:
    exprs += [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulo"),
        F.sum(F.when(texto(c) == "", 1).otherwise(0)).alias(f"{c}__vazio"),
        F.sum(F.when(F.upper(texto(c)).isin(TOKENS_NULOS), 1).otherwise(0)).alias(f"{c}__texto_nulo"),
    ]
r = df_pesquisa.agg(*exprs).first()

df_pesquisa_faltantes = (
    spark.createDataFrame(
        [(c, r[f"{c}__nulo"], r[f"{c}__vazio"], r[f"{c}__texto_nulo"]) for c in df_pesquisa.columns],
        "coluna string, nulo long, vazio long, texto_nulo long",
    )
    .withColumn("total_faltante", F.col("nulo") + F.col("vazio") + F.col("texto_nulo"))
    .withColumn("pct_faltante", F.round(F.col("total_faltante") * 100 / total, 2))
    .orderBy(F.desc("pct_faltante"))
)
mostrar(df_pesquisa_faltantes,
        "Faltantes por coluna: null real, string vazia e texto tipo 'NULL', com % sobre o total de linhas")

### 3. nota_atendimento x escala de 1 a 5

In [0]:
n = texto("nota_atendimento")
df_pesquisa_nota = df_pesquisa.withColumn(
    "categoria_nota",
    F.when(eh_faltante("nota_atendimento"), "faltante")
     .when(n.rlike(r"^[1-5]$"), "ok: inteiro de 1 a 5")
     .when(n.rlike(r"^-?\d+$"), "inteiro fora da escala")
     .when(n.rlike(r"^-?\d+[.,]\d+$"), "decimal")
     .otherwise("não numérico"),
)
mostrar(
    df_pesquisa_nota.groupBy("categoria_nota").count()
          .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
          .orderBy(F.desc("count")),
    "Distribuição de nota_atendimento por categoria",
)
mostrar(
    df_pesquisa_nota.groupBy("categoria_nota", "nota_atendimento").count()
          .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
          .orderBy("categoria_nota", "nota_atendimento"),
    "Valores de nota_atendimento (grafia original) por categoria",
)

### 4. Integridade com chamados

In [0]:
ids_chamados = (spark.table(f"{bronze_schema}.chamados")
                  .filter(~eh_faltante("id_chamado"))
                  .select(texto("id_chamado").alias("id_chamado")).distinct())
ids_pesquisa = (df_pesquisa.filter(~eh_faltante("id_chamado"))
                  .select(texto("id_chamado").alias("id_chamado")).distinct())

pesquisas_sem_chamado = ids_pesquisa.join(ids_chamados, on="id_chamado", how="left_anti")
n_chamados = ids_chamados.count()
n_com_pesquisa = ids_pesquisa.count() - pesquisas_sem_chamado.count()

print(f"id_chamado em pesquisa_satisfacao que não existem em chamados: {pesquisas_sem_chamado.count()}")
print(f"Chamados com pesquisa: {n_com_pesquisa} de {n_chamados} ({round(n_com_pesquisa * 100 / n_chamados, 2)}%)")
mostrar(pesquisas_sem_chamado.orderBy("id_chamado"), "id_chamado de pesquisas sem chamado correspondente")

### Conclusões e tratamentos previstos para a Silver

| Problema encontrado | Evidência | Tratamento na Silver |
|---|---|---|
| Pesquisa duplicada com outro `id_pesquisa` | 802 `id_chamado` com 2 pesquisas, sempre com a mesma nota e `id_pesquisa` diferente. Por isso não aparecem como cópia exata | Deduplicar por `id_chamado`, mantendo o menor `id_pesquisa` |
| `nota_atendimento` faltante | 2.842 vazios e 2.836 textos `NULL` no CSV (cerca de 7%) | Converter para null |
| Notas fora da escala de 1 a 5 | 421 com -1, 393 com 0 e 407 com 6 | Converter para null |
| Tipo da nota | Lida como texto | Cast para inteiro após os tratamentos acima |
| `id_pesquisa` e `id_chamado` | Sem faltantes e sem formato inválido | Nenhum |
| Integridade com chamados | Todos os `id_chamado` existem em chamados. 80.219 de 131.838 chamados (60,85%) têm pesquisa | Nenhum. Chamado sem pesquisa é esperado (cliente não respondeu) |